# Tutorial: Foundational Neural Quantum States with Disorder
## Training & Testing a ViT-based ansatz on the disordered Ising model

---

### Learning objectives

By the end of this tutorial you will be able to:

1. **Understand the physics** — define a 1D transverse-field Ising model with a uniform but random global field $h_0$.
2. **Build a Foundational NQS** — use `netket_foundational` (nkf) to train a single ViT ansatz simultaneously on many disorder realizations.
3. **Run the optimization** — set up a Natural-Gradient VMC loop with `nkf.VMC_NG`.
4. **Evaluate the results** — compare energies against exact diagonalization (ED).
5. **Diagnose quality** — plot the **V-score** and **R̂ (Rhat)** convergence diagnostics.

---

### Prerequisites

```
pip install netket netket_foundational flax optax einops scipy matplotlib pandas tqdm
```

This notebook runs comfortably on a single GPU (tested on A100). For CPU-only runs, reduce L, N and n_iter as suggested in the comments.

---
## Part 0 — Background

### 0.1  The transverse-field Ising model with a random global field

We study the 1D quantum Ising chain with a **uniform but randomly drawn transverse field** $h_0$:

$$
H(h_0) = -h_0 \sum_{i=1}^{L} \sigma^x_i  \;-\; J\sum_{i=1}^{L} \sigma^z_i\,\sigma^z_{i+1}
$$

with periodic boundary conditions. Each disorder **realization** is a single scalar $h_0 \sim \mathcal{U}(0, h_{\max})$, shared by all sites.  
In the code, the parameter vector passed to the network is the constant vector $\mathbf{h} = (h_0, h_0, \dots, h_0) \in \mathbb{R}^L$ — all entries are equal.  
The scalar $h_0$ is the **Hamiltonian parameter** our ansatz is conditioned on: we want to learn $|\psi_0(h_0)\rangle$ for many values of $h_0$ simultaneously.

### 0.2  Foundational NQS — the key idea

A *standard* NQS $\psi_\theta(\boldsymbol{\sigma})$ parametrizes one wave function.  
A **Foundational NQS** extends this to a *family* of wave functions:

$$
\psi_\theta(\boldsymbol{\sigma};\, \mathbf{h})
$$

The Hamiltonian parameter $h_0$ is broadcast into a constant vector and concatenated to the spin configuration before being fed to the network, so the same weights $\theta$ serve all values of $h_0$ simultaneously.  
Training is done jointly over $N$ replicas, each with its own $h_0^{(r)}$.

### 0.3  The ViT ansatz

ViTFNQS uses a Vision-Transformer-like encoder:

```
spins (L,)  ──┐
               ├─► patch & embed ─► Encoder (L attention blocks) ─► OutputHead ─► log ψ ∈ ℂ
params (n,) ──┘
```

Each spin-patch and the disorder parameters are concatenated and linearly embedded into tokens of dimension `d_model`. The attention mechanism (`FMHA`) then mixes information across tokens.

---
## Part 1 — Setup

In [ ]:
import os
# Enable NetKet experimental sharding (multi-device parallelism)
os.environ["NETKET_EXPERIMENTAL_SHARDING"] = "1"
# Prevent JAX from pre-allocating a la rge part of GPU memory
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from tqdm import tqdm
import optax
import netket as nk
import netket_foundation as nkf
from netket_foundation._src.model.vit import ViTFNQS
from flax import serialization
from netket.sampler import rules
from netket.utils import struct

print("JAX devices:", jax.devices())
print("NetKet version:", nk.__version__)

---
## Part 2 — Physical system and disorder generation

### 2.1  Hilbert space and parameter space

Block 1 : ParameterSpace: tells netket foundational the shape and range (here from 0 to $h_0$ ) of the Hamiltonian parameters. Notice that even though the physical parameter is a single scalar h0, we pass it as a constant vector of size L (one entry per site, all equal to h0)

In [ ]:
# Physical parameters 
L    = 16      # System size  (reduce to 8 for CPU-only runs)
h0   = 1.0     # Uniform upper bound on the random fields
J    = 1.0     # Ising coupling (here J = 1/e ≈ 0.368)
               # we use J = 1.0 for simplicity)
N    = 20      # Number of disorder realizations trained simultaneously
               # (reduce to 8 on CPU)

seed = 42
rng  = np.random.default_rng(seed)
k    = jax.random.key(seed)

# Hilbert space: L spin-1/2 sites
hi = nk.hilbert.Spin(0.5, L)
print(f"Hilbert space dimension: 2^{L} = {hi.n_states}")

ps = nkf.ParameterSpace(N=hi.size, min=0, max=h0)
print(f"Parameter space dimension: {ps.size}  (= L = {hi.size}, all entries = h0)")

### 2.2  Generating disorder realizations


In [ ]:
def generate_disorder(N_real, system_size, h_max, rng=None):
    """Return an array of shape (N_real, system_size).
    Each realization draws a single scalar h0 ~ U(0, h_max),
    then broadcasts it to all L sites: params[r] = [h0, h0, ..., h0].
    """
    if rng is None:
        rng = np.random.default_rng()
    h0_values = rng.uniform(0.0, h_max, size=N_real)   # one scalar per realization
    return np.tile(h0_values[:, None], (1, system_size))  # broadcast to (N_real, L)

params_train = generate_disorder(N, hi.size, h0, rng=rng)
print(f"Training disorder realizations shape: {params_train.shape}")
print("First realization — all entries equal to h0 =", params_train[0, 0].round(3))
assert np.allclose(params_train[0], params_train[0, 0]), "All sites must share the same h0"

You can visualize the distribution of the values, as well as a heatmap of the realizations shown above. The exact plotting code is not critical for the remainder of the analysis.

In [ ]:
# Visualization: uniform disorder realizations
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

#heatmap of the N realizations (value of h0 per site)
ax = axes[0]
im = ax.imshow(params_train, aspect="auto", cmap="viridis", vmin=0, vmax=h0)
plt.colorbar(im, ax=ax, label=r"$h_0^{(r)}$")
ax.set_xlabel("Site $i$", fontsize=11)
ax.set_ylabel("Realization $r$", fontsize=11)
ax.set_title("Heatmap of disorder parameters\n"
             "(all columns are identical: uniform $h_0$ per realization)", fontsize=10)

# distribution of scalar values h0
ax = axes[1]
h0_scalars = params_train[:, 0]   # one scalar per realization
ax.hist(h0_scalars, bins=10, color="steelblue", edgecolor="white", alpha=0.9)
ax.axvline(h0_scalars.mean(), color="red", linestyle="--", label=f"Mean = {h0_scalars.mean():.2f}")
ax.set_xlabel(r"$h_0^{(r)}$ (scalar value)", fontsize=11)
ax.set_ylabel("Number of realizations", fontsize=11)
ax.set_title(f"Distribution of {N} realizations\n"
             r"$h_0 \sim \mathcal{U}(0," + f"{h0})$", fontsize=10)
ax.legend()

fig.tight_layout()
plt.show()

Furthermore, we can also generate disorder realizations according to a Gaussian distribution. In contrast to the previous case, different sites will have distinct values of the local magnetic field. This provides a good model for the screening of the magnetic field by the local configuration of the site.

We note that if our h0_list contains values of h0_list
that are too close to zero, and a sample drawn from a Gaussian distribution centered around the smallest h0
yields a negative value, we symmetrize it by taking its absolute value.

This code takes as input a list of $h_0$ values, corresponding to transverse-field expectation values, as well as a number n_reps of disorder realizations. For each $h_0$, n_reps disorder configurations are sampled from a Gaussian distribution with mean $h_0$. In addition to this, the homogeneous (disorder-free) configuration, where all sites have the same transverse field, is included.

In [ ]:
def generate_multi_h0_disorder(h0_list, n_reps, system_size, sigma, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    
    all_configs = []
    
    for h_m in h0_list:
        
        # We first sample from the Gaussian distribution (which may include negative values)
        raw_configs = rng.normal(loc=h_m, scale=sigma, size=(n_reps, system_size))
        
        # Negative values are “reflected” back into positive values, preserving a smooth distribution.
        random_configs = np.abs(raw_configs)
        
        # Creation of the homogenous configuration(h_m, h_m, ..., h_m)
        
        homogeneous_config = np.full((1, system_size), h_m)
        
        # we stack them : we get (n_reps + 1) configurations for this h_m
        batch_configs = np.vstack([random_configs, homogeneous_config])
        
        all_configs.append(batch_configs)
        
    return np.vstack(all_configs)

Similarly, you can visualize the distribution using this new method for generating the disorder configuration. This code compares the uniform and the gaussian disorder. This code is only a tool of visualization, the details of the code are unrelevant to the study.

In [ ]:
h0_list_demo = [0.2, 0.5, 0.8, 1.0]
n_reps_demo  = 15
sigma_demo   = 0.08

params_gauss = generate_multi_h0_disorder(h0_list_demo, n_reps_demo, L, sigma_demo, rng=rng)
print(f"Shape of Gaussian realizations: {params_gauss.shape}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Block 1: heatmap of Gaussian disorder (site-dependent)
ax = axes[0]
im = ax.imshow(params_gauss, aspect="auto", cmap="plasma")
plt.colorbar(im, ax=ax, label=r"$h_i^{(r)}$")
ax.set_xlabel("Site $i$", fontsize=11)
ax.set_ylabel("Realization $r$", fontsize=11)

# Separators between groups of h0
for g in range(1, len(h0_list_demo)):
    ax.axhline(g * (n_reps_demo + 1) - 0.5, color="white", linewidth=1.5, linestyle="--")

# Labels for the groups
for g, hm in enumerate(h0_list_demo):
    y_mid = g * (n_reps_demo + 1) + n_reps_demo / 2
    ax.text(-0.5, y_mid, f"$h_0={hm}$", ha="right", va="center", fontsize=8,
            color="black", transform=ax.get_yaxis_transform())

ax.set_title(f"Gaussian disorder ($\\sigma={sigma_demo}$)\n"
             "Each site has a different value", fontsize=10)

# Block 2: distribution by h0 value
ax = axes[1]
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(h0_list_demo)))
for g, hm in enumerate(h0_list_demo):
    start = g * (n_reps_demo + 1)
    end   = start + n_reps_demo
    vals  = params_gauss[start:end].flatten()
    ax.hist(vals, bins=20, alpha=0.6, color=colors[g], label=f"$h_0={hm}$", density=True)
    ax.axvline(hm, color=colors[g], linewidth=2, linestyle="--")

ax.set_xlabel(r"Value of $h_i^{(r)}$", fontsize=11)
ax.set_ylabel("Density", fontsize=11)
ax.set_title("Distribution of local fields\n"
             "(dashed lines = central $h_0$ values)", fontsize=10)
ax.legend(fontsize=9)

fig.tight_layout()
plt.show()

The remainder of this study will employ Gaussian disorder rather than the uniform distribution, which was introduced solely for the purpose of presenting the topic.

### 2.3  Defining the Hamiltonian

We use nkf.operator.ParametrizedOperator , which wraps a constructor function create_operator that takes a parameter vector and returns a NetKet operator.

ParametrizedOperator takes as arguments the parameter space ps, the operator, and the Hilbert space. This allows the operator to be constructed on demand: whenever the user needs to evaluate it (for instance, to compute an energy), they call create_operator(parameters) with the current parameters to obtain the corresponding concrete operator. 

In [ ]:
def create_operator(params):
    
    assert params.shape == (hi.size,), f"Expected shape ({hi.size},), got {params.shape}"
    h0_val = params[0]   # all entries are equal, we just read the first one

    # Transverse-field term:  -h0 ∑_i σ^x_i
    ha_X = h0_val * sum(
        nkf.operator.sigmax(hi, i)
        for i in range(hi.size)
    )

    # Ising interaction:  -J ∑_i σ^z_i σ^z_{i+1}   (periodic BC)
    ha_ZZ = sum(
        nkf.operator.sigmaz(hi, i) @ nkf.operator.sigmaz(hi, (i + 1) % hi.size)
        for i in range(hi.size)
    )

    return -ha_X - J * ha_ZZ

ha_p  = nkf.operator.ParametrizedOperator(hi, ps, create_operator)

# Magnetization observable  Mz = (1/L) ∑_i σ^z_i
Mz    = sum(nkf.operator.sigmaz(hi, i) for i in range(hi.size)) * (1.0 / hi.size)
mz_p  = nkf.operator.ParametrizedOperator(hi, ps, lambda _: Mz)

# Quick sanity check: build the operator for the first realization
H0 = create_operator(params_train[0])
print("Operator built for first realization:", H0)

Here, to gain familiarity with these operators, you can use the code below to plot the first eigenvalues of the Hamiltonian as a function of h0. Nous utilisons create_operator pour créer l'hamiltonien correspondant à la valeur de $h_0$ voulue.

In [ ]:
import scipy.linalg

L_sp   = 6      
hi_sp  = nk.hilbert.Spin(0.5, L_sp)
h0_sweep = np.linspace(0.0, 2.0, 40)
n_levels = 4         

energies_sweep = []

for h_val in h0_sweep:
    # Construction of H using the same logic as create_operator
    ha_X_sp  = h_val * sum(nkf.operator.sigmax(hi_sp, i) for i in range(L_sp))
    ha_ZZ_sp = sum(
        nkf.operator.sigmaz(hi_sp, i) @ nkf.operator.sigmaz(hi_sp, (i+1) % L_sp)
        for i in range(L_sp)
    )
    H_mat = (-ha_X_sp - J * ha_ZZ_sp).to_dense()
    evals = np.sort(np.linalg.eigvalsh(H_mat))[:n_levels]
    energies_sweep.append(evals)

energies_sweep = np.array(energies_sweep)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# energy levels vs h0
ax = axes[0]
labels = ["GS $E_0$", "$E_1$", "$E_2$", "$E_3$"]
colors_sp = ["royalblue", "tomato", "seagreen", "darkorange"]
for lvl in range(n_levels):
    ax.plot(h0_sweep, energies_sweep[:, lvl], color=colors_sp[lvl],
            linewidth=2 if lvl == 0 else 1, label=labels[lvl])
ax.axvline(J, color="gray", linestyle=":", label=f"$h_0=J={J}$ (QPT)")
ax.set_xlabel(r"$h_0 / J$", fontsize=12)
ax.set_ylabel("Energy", fontsize=12)
ax.set_title(f"Exact spectrum of $H(h_0)$ for $L={L_sp}$\n"
             "(quantum phase transition at $h_0 = J$)", fontsize=10)
ax.legend(fontsize=9)

# gap (E1-E0) vs h0
ax = axes[1]
gap = energies_sweep[:, 1] - energies_sweep[:, 0]
ax.plot(h0_sweep, gap, color="purple", linewidth=2)
ax.axvline(J, color="gray", linestyle=":", label=f"$h_0=J={J}$")
ax.set_xlabel(r"$h_0 / J$", fontsize=12)
ax.set_ylabel(r"Gap $E_1 - E_0$", fontsize=12)
ax.set_title(f"Energy gap vs $h_0$ ($L={L_sp}$)\n"
             "The gap closes at the phase transition", fontsize=10)
ax.legend(fontsize=9)

fig.tight_layout()
plt.show()
print(f"Critical point (minimum gap): h0 ≈ {h0_sweep[np.argmin(gap)]:.2f}")

---
## Part 3 — Building the Foundational NQS

### 3.1  The ViTFNQS model

Key hyper-parameters:

| Parameter | Role |
|-----------|------|
| `b` | patch size — spins are grouped in windows of size `b` before embedding |
| `L_eff` | effective sequence length = `L // b` |
| `d_model` | token embedding dimension |
| `heads` | number of attention heads in FMHA |
| `num_layers` | depth of the Transformer encoder |
| `n_coups` | size of the disorder parameter vector appended to each token |
| `disorder=True` | uses a dedicated embedding for the disorder parameters |
| `transl_invariant=False` | disorder breaks translation symmetry, so we disable it |
| `complex=True` | the ansatz outputs a complex amplitude $\log\psi \in \mathbb{C}$ |

In this part we encode the foundational neural quantum state in the form of a vision transformer, with the parameters listed above. First we construct a transformer with the desired parameters  

In [ ]:
b     = 4         # patch size   (must divide L)
L_eff = L // b    # effective sequence length

ma = ViTFNQS(
    num_layers       = 2,       # number of Transformer encoder blocks
    d_model          = 16,      # embedding dimension
    heads            = 4,       # attention heads
    b                = b,       # patch size
    L_eff            = L_eff,   # effective sequence length
    n_coups          = ps.size, # appended disorder vector length
    complex          = True,    # complex-valued ansatz
    disorder         = True,    # use disorder-specific embedding
    transl_invariant = False,   # no translation symmetry (disorder breaks it)
    two_dimensional  = False,   # 1D chain
)

print("Model architecture summary:")
print(f"  Patch size b      = {b}")
print(f"  Effective length  = {L_eff}  tokens")
print(f"  Token dimension   = {ma.d_model}")
print(f"  Disorder concat   = {ma.n_coups} params per token")

### 3.2  Sampler and Foundational Quantum State

`FoundationalQuantumState` (nkf) is the core object:  
it wraps a single set of neural-network weights **plus** an array of $N$ disorder realizations and runs MCMC sampling and gradient estimation simultaneously for all replicas. This is were the foundational aspect of the method takes place. In our case, each replica in going to correspond to a different realization of disorder (ie a different hamiltonien) and the transformer is going to be optimized on all simultaniously. 

In this bloc we define the parameters of the MCMC optimization. n_chains_per_replica and n_samples are similar to what is found in classical NQS, but multiplied by the number of replicas. sa defines the sampler heuristic, for example MetropolisLocal allows one spin-flip at each step. 

In [ ]:
#Sampler
# # MetropolisLocal proposes single-spin flips — a good default for spin systems.
n_chains_per_replica = 16
n_chains = N * n_chains_per_replica          # total chains across all replicas
n_samples = N * n_chains_per_replica * 4     # 4 samples per chain per step

sa = nk.sampler.MetropolisLocal(hi, n_chains=n_chains)

#Variational state
vs = nkf.FoundationalQuantumState(
    sa,
    ma,
    ps,
    n_replicas = N,
    n_samples  = n_samples,
    seed       = seed,
)

# Assign disorder realizations
vs.parameter_array = params_train
print(f"FoundationalQuantumState ready: {N} replicas, {n_samples} total samples/step")

To illustrate sampling via Monte Carlo chains, the code below shows how the magnetization evolves as a function of the configurations explored during the Monte Carlo sampling. This bloc has no optimization in it, it simply shoes how a transformer with random weights acts on different configurations, modified by the Metropolis-Hastings algorithm.

In [ ]:
# Visualization: MCMC chain evolution before training
# We track how the instantaneous magnetization ⟨σz⟩ evolves
# along each chain

n_steps_show  = 30
n_chains_show = 6

_pars      = params_train[0]
_vs_single = vs.get_state(_pars)

vs_mc_viz = nk.vqs.MCState(
    sampler    = nk.sampler.MetropolisLocal(hi, n_chains=n_chains_show),
    model      = _vs_single.model,
    variables  = _vs_single.variables,
    n_samples  = n_chains_show,
    chunk_size = n_chains_show,
)

# Collect magnetizations at each step 
mz_traces = []
for _ in range(n_steps_show):
    samples = vs_mc_viz.samples                        # shape (n_chains_show, L)
    mz_inst = np.mean(np.array(samples), axis=-1)     # ⟨σz⟩ per chain
    mz_traces.append(mz_inst)
    vs_mc_viz._samples = None                          # invalidate cache, preserve sampler_state

mz_traces = np.array(mz_traces)   # (n_steps_show, n_chains_show)

fig, ax = plt.subplots(figsize=(10, 4))
for c in range(n_chains_show):
    ax.plot(mz_traces[:, c], alpha=0.7, label=f"Chain {c+1}")
ax.axhline(0, color="black", linestyle="--", linewidth=0.8)
ax.set_xlabel("MCMC step", fontsize=11)
ax.set_ylabel(r"Instantaneous $\langle \sigma^z \rangle$", fontsize=11)
ax.set_title(f"Magnetization evolution over {n_chains_show} MCMC chains\n"
             f"(untrained initial state, first realization: $h_0={_pars[0]:.2f}$)", fontsize=10)
ax.legend(ncol=3, fontsize=8)
fig.tight_layout()
plt.show()

---
## Part 4 — Optimization

We use **Natural Gradient VMC** (nkf.VMC_NG) with a linear learning-rate schedule and a diagonal-shift regularization of the quantum geometric tensor.

VMC_NG computes the quantum natural gradient efficiently using the stochastic reconfiguration method. The diag_shift parameter adds $\epsilon\,\mathbf{I}$ to the QFI matrix to avoid singularities.

Code Explanation:

Block 1:
The linear_schedule function from optax enables a linear adjustment of the learning rate over the course of the iterations. The optimizer used is based on natural gradient descent.

Block 2:
The Foundational object from NetKet takes as input the parameterized Hamiltonian defined in Part 2, the natural gradient descent optimizer, and the variational state.

In [ ]:
n_iter     = 300     # number of optimization steps (reduce to 100 on CPU)
lr_init    = 0.03
lr_end     = 0.005
diag_shift = 1e-4


# Block 1
learning_rate = optax.linear_schedule(
    init_value      = lr_init,
    end_value       = lr_end,
    transition_steps= n_iter,
)
optimizer = optax.sgd(learning_rate)
#Block 2
gs = nkf.VMC_NG(
    ha_p,
    optimizer,
    variational_state = vs,
    diag_shift = diag_shift,
)

os.makedirs("output", exist_ok=True)
log = nk.logging.JsonLog("output/log", save_params=True)

print("Optimizer set up. Starting training...")

The parameter obs in the block below contains the observables that are measured at each iteration step.

In [ ]:
# Run 
gs.run(
    n_iter,
    out  = log,
    obs  = {"ham": ha_p, "mz": mz_p},
    step_size = 10,   # log every 10 steps
)
print("Training complete.")

---
## Part 5 — Convergence diagnostics

### 5.1  Energy convergence curves

The `log.data` dictionary stores, for each observable, one time-series per replica.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

for r in range(N):
    e_log = log.data["Energy"]      # shape: (n_logged_steps, N)
    # log.data stores each replica separately; index by replica r
    replica_log = log.data["ham"][r]
    ax.plot(replica_log.iters, np.real(replica_log.Mean), alpha=0.4, linewidth=0.8)

ax.set_xlabel("Iteration")
ax.set_ylabel("Energy")
ax.set_title("Training energy — all disorder realizations")
ax.set_xscale("log")
fig.tight_layout()
plt.savefig("output/convergence.pdf")
plt.show()

### 5.2  R̂ (Rhat) — MCMC chain convergence
 
R̂ (Gelman–Rubin statistic) measures whether multiple independent MCMC chains have converged to the same distribution. It compares the *within-chain* variance to the *between-chain* variance:

$$
\hat{R} = \sqrt{\frac{\hat{V}}{W}}
$$

where $W$ is the mean within-chain variance and $\hat{V}$ is an estimate of the true variance using the pooled chains.  

- $\hat{R} \approx 1$ → chains have mixed well 
- $\hat{R} > 1.05$ → chains have not converged, increase n_samples or n_chains.

NetKet stores R̂ automatically in the Stats object returned by vs.expect().

The get_state method returns the state corresponding to the parameters pars.

In [ ]:
# Compute R̂ for each replica on the final variational state
rhat_values = []

for r in tqdm(range(N), desc="Computing R̂"):
    pars = vs.parameter_array[r
    _vs  = vs.get_state(pars)          

    # Create a small MCState for this replica
    vs_mc = nk.vqs.MCState(
        sampler   = nk.sampler.MetropolisLocal(hi, n_chains=32),
        model     = _vs.model,
        variables = _vs.variables,
        n_samples = 512,
        chunk_size= 64,
    )

    stats = vs_mc.expect(create_operator(pars))
    rhat_values.append(float(np.real(stats.R_hat)))

rhat_values = np.array(rhat_values)

fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(range(N), rhat_values, color="steelblue", alpha=0.8)
ax.axhline(1.1, color="red", linestyle="--", label="R̂ = 1.1 threshold")
ax.set_xlabel("Replica index")
ax.set_ylabel("R̂")
ax.set_title("Gelman–Rubin R̂ per disorder realization")
ax.legend()
fig.tight_layout()
plt.savefig("output/rhat.pdf")
plt.show()

print(f"Mean R̂ = {rhat_values.mean():.4f}  |  Max R̂ = {rhat_values.max():.4f}")
n_bad = (rhat_values > 1.1).sum()
print(f"Replicas with R̂ > 1.1 : {n_bad}/{N}")

When evaluating an operator in a quantum state, it is necessary to resample in order to approximate the measurement of the quantum state of interest. We then use samples from the Markov chains to compute an average over the number of samples evaluated with the operator. However, if the initial samples are of poor quality, they may bias the measurement. To address this, one can introduce a parameter that discards a certain number of the initial samples, namely n_burning_chains.

In [ ]:
vs_mc = nk.vqs.MCState(
        sampler   = nk.sampler.MetropolisLocal(hi, n_chains=32),
        model     = _vs.model,
        variables = _vs.variables,
        n_samples = 512,
        chunk_size= 64,
        #does not take into account the first 500 samples generated by the sampler
        n_burning_chains=500,

)

We can visualize the burning effect by comparing energy estimates with and without burn-in (on the last realization) and comparing it also with the real energy value that we compute thanks to a FullSumState that we are going to discover later.

This code performs one iteration for each value of the index n_burn, and then carries out n_estimates iterations in order to average the computed energies over these n_estimates runs. Within this inner loop, an MCState is used to estimate the energy of the most recently obtained state.

This code is relevant because the 256 samples are only taken into account once the number of samples in the Markov chain exceeds n_burn. Therefore, when n_burn is set to 0, the initial samples—typically of lower quality due to random initialization—are included in the estimation. As n_burn increases, samples are incorporated only once they have reached a higher quality, leading to a more accurate estimation of the energy with respect to the exact value.

In [ ]:
#Visualization: effect of burn-in on energy estimation
n_burn_values = [0, 50, 100, 200, 500]
n_estimates   = 8     # repetitions per burn-in value
_ha_viz = create_operator(_pars)

mean_energies = []
std_energies  = []

for n_burn in n_burn_values:
    Es = []
    for _ in range(n_estimates):
        vs_b = nk.vqs.MCState(
            sampler          = nk.sampler.MetropolisLocal(hi, n_chains=32),
            model            = _vs_single.model,
            variables        = _vs_single.variables,
            n_samples        = 256,
            chunk_size       = 64,
            n_discard_per_chain = n_burn,
        )
        stat = vs_b.expect(_ha_viz)
        Es.append(float(np.real(stat.Mean)))
    mean_energies.append(np.mean(Es))
    std_energies.append(np.std(Es))

mean_energies = np.array(mean_energies)
std_energies  = np.array(std_energies)

fig, ax = plt.subplots(figsize=(8, 4))
ax.errorbar(n_burn_values, mean_energies, yerr=std_energies,
            fmt="o-", color="steelblue", capsize=5, linewidth=2, label="Estimated energy ± standard deviation")
E_ref = float(np.real(nk.vqs.FullSumState(hi, _vs_single.model, _vs_single.variables, chunk_size=64)
              .expect(_ha_viz).Mean))
ax.axhline(E_ref, color="red", linestyle="--", label=f"Exact energy (FullSum) = {E_ref:.4f}")
ax.set_xlabel("Number of burn-in samples (`n_discard_per_chain`)", fontsize=11)
ax.set_ylabel("Energy", fontsize=11)
ax.set_title("Impact of burn-in on the accuracy of energy estimation", fontsize=11)
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

Furthermore, it is possible that your R̂ value is too large, indicating an issue with the sampling of the Markov chains. This problem may arise both during the evaluation of observables and during training. Several factors may be responsible: either the chains fail to converge to the target distribution, or they exhibit excessively strong intra-chain correlations.

First, to improve independence within the chains:

In [ ]:
#The sweep size is increased. If it is equal to 3, the sampler performs 3 iterations between each sample
sa = nk.sampler.MetropolisLocal(hi, n_chains=n_chains, sweep_size=3)

To improve the convergence of the sampling chains, we ensure that they explore the entire space of possible spin configurations.

We can therefore modify the sampling rule: in this GlobalFlipRule, we introduce a uniform random variable that determines, at each sampling step, whether to apply the standard sampling rule or to flip all spins simultaneously.


In [ ]:
class GlobalFlipRule(rules.MetropolisRule):
    
    prob_global: float = struct.field(pytree_node=False)

    def __init__(self, prob_global=0.1):
        self.prob_global = prob_global

    def transition(self, sampler, machine, parameters, state, key, sigma):
        n_chains = sigma.shape[0]
        L = sigma.shape[-1]
        
        key_prob, key_site = jax.random.split(key, 2)

        # Propositions
        sigma_global = -sigma
        sites = jax.random.randint(key_site, shape=(n_chains,), minval=0, maxval=L)
        mask = jax.nn.one_hot(sites, L)
        sigma_local = sigma * (1 - 2 * mask)

        # Choice
        rand_vals = jax.random.uniform(key_prob, shape=(n_chains, 1))
        sigma_prop = jnp.where(rand_vals < self.prob_global, sigma_global, sigma_local)

        return sigma_prop.astype(sigma.dtype), None

In the main code, we can then replace the line involving the sampler sa with:

In [ ]:
prob_global_flip=0.01
sa = nk.sampler.MetropolisSampler(
    hi,
    rule=GlobalFlipRule(prob_global_flip),
    n_chains=n_chains
)

Finally, another way to ensure that the sampler fully explores the configuration space and does not become trapped in a specific region, or in states that are difficult to reach, is to properly initialize it.

Indeed, up to now, the initial configuration of the sampler has been randomly chosen. We now choose to initialize half of the Markov chains in the fully “up” spin configuration, and the other half in the fully “down” configuration.

Code explanation:

Block 1:
The original sampler state is retrieved, i.e. the exact state array generated by NetKet (which contains both spins and couplings).

Block 2:
Only the first L columns, corresponding to the spins, are modified in order to ensure that the couplings remain unchanged. Half of the chains are initialized to 1, while the other half are initialized to -1.

Block 3:
The sampler is then updated with the modified state.

In [ ]:
#After the definition of vs and sa

sa = nk.sampler.MetropolisSampler(
    hi,
    rule=GlobalFlipRule(prob_global_flip),
    n_chains=n_chains
)
vs = nkf.FoundationalQuantumState(sa, ma, ps, n_replicas=total_configs_train, n_samples=n_samples, seed=seed, chunk_size=chunk_size)
# Block 1
sigma_orig = vs.sampler_state.σ


flat_sigma = sigma_orig.reshape(-1, sigma_orig.shape[-1])
half = flat_sigma.shape[0] // 2


# Block 2
flat_sigma = flat_sigma.at[:half, :L].set(1)

flat_sigma = flat_sigma.at[half:, :L].set(-1)

#Block 3

sigma_new = flat_sigma.reshape(sigma_orig.shape)
vs.sampler_state = vs.sampler_state.replace(σ=sigma_new)


---
## Part 6 — Comparison with exact diagonalization

For $L \le 20$ we can compute the exact ground-state energy with NetKet's Lanczos solver and compare it to our VMC estimate. in this first bloc, for each realization of disorder, we calculate the exact ground state energy with the lanczos solver, and then we calculate the ground energy of the state encoded by our transformer. To have a more precise result, we use FullSumState rather than an MCMC algorithm to estimate the energy of the FNQS (only possible for small systems)

In [ ]:
exact_energies = []
vmc_energies   = []
rel_errors     = []

for r, pars in tqdm(enumerate(vs.parameter_array), total=N, desc="ED comparison"):
    _ha  = create_operator(pars)
    E_ed = nk.exact.lanczos_ed(_ha, k=1, compute_eigenvectors=False).item()

    # VMC energy: use the full-sum state for an unbiased estimate (works for L≤20)
    _vs = vs.get_state(pars)
    vs_fs = nk.vqs.FullSumState(
        hilbert    = hi,
        model      = _vs.model,
        variables  = _vs.variables,
        chunk_size = 64,
    )
    E_vmc = float(np.real(vs_fs.expect(_ha).Mean))

    exact_energies.append(E_ed)
    vmc_energies.append(E_vmc)
    rel_errors.append(abs((E_vmc - E_ed) / E_ed))

exact_energies = np.array(exact_energies)
vmc_energies   = np.array(vmc_energies)
rel_errors     = np.array(rel_errors)

print(f"Mean relative energy error : {rel_errors.mean()*100:.3f} %")
print(f"Max  relative energy error : {rel_errors.max()*100:.3f} %")

Here we simply plot our results 

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Panel 1 : scatter VMC vs ED 
ax = axes[0]
ax.scatter(exact_energies, vmc_energies, alpha=0.7, s=30, label="VMC vs ED")
lims = [min(exact_energies.min(), vmc_energies.min()),
        max(exact_energies.max(), vmc_energies.max())]
ax.plot(lims, lims, "k--", linewidth=1, label="identity")
ax.set_xlabel("Exact energy $E_0$")
ax.set_ylabel("VMC energy")
ax.set_title("VMC vs exact diagonalization")
ax.legend()

# Panel 2 : relative error histogram 
ax = axes[1]
ax.hist(rel_errors * 100, bins=15, color="steelblue", edgecolor="white")
ax.set_xlabel("Relative error (%)")
ax.set_ylabel("Count")
ax.set_title("Distribution of relative energy errors")

fig.tight_layout()
plt.savefig("output/ed_comparison.pdf")
plt.show()

---
## Part 7 — V-score diagnostic


The **V-score** (Variational score) is a dimensionless, size-intensive measure of ansatz quality:

$$
V = \frac{\text{Var}[H]}{E_0^2} = \frac{\langle H^2\rangle - \langle H\rangle^2}{\langle H\rangle^2}
$$

$V = 0$ → exact ground state   
Larger $V$ → larger variational energy uncertainty 

It is particularly useful for **comparing different ansätze** across system sizes, since it normalizes by the energy scale.

A V-score below $10^{-3}$ is generally considered satisfying for ground-state problems.

In this bloc, we calculate the V-score for each realization of disorder. This time, we use mc_expect rather than FullSumState because this method needs to work for bigger systems. We then calculate the mean of all V-score and show the biggest one

In [ ]:
v_scores = []

for r, pars in tqdm(enumerate(vs.parameter_array), total=N, desc="V-score"):
    _vs = vs.get_state(pars)

    vs_mc = nk.vqs.MCState(
        sampler   = nk.sampler.MetropolisLocal(hi, n_chains=32),
        model     = _vs.model,
        variables = _vs.variables,
        n_samples = 2048,
        chunk_size= 64,
    )

    stats = vs_mc.expect(create_operator(pars))
    E     = float(np.real(stats.Mean))
    Var   = float(np.real(stats.Variance))
    vscore = Var / (E**2 + 1e-12)      # small epsilon to avoid division by zero
    v_scores.append(vscore)

v_scores = np.array(v_scores)
print(f"Mean V-score : {v_scores.mean():.2e}")
print(f"Max  V-score : {v_scores.max():.2e}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(N), v_scores, color="darkorange", alpha=0.8)
ax.axhline(1e-3, color="red", linestyle="--", label="V-score = 10⁻³")
ax.set_yscale("log")
ax.set_xlabel("Replica index")
ax.set_ylabel("V-score")
ax.set_title("V-score per disorder realization")
ax.legend()
fig.tight_layout()
plt.savefig("output/vscore.pdf")
plt.show()

---
## Part 8 — Testing on unseen disorder realizations

A key advantage of the Foundational NQS is **zero-shot generalization**:  
we can evaluate the trained model on disorder realizations it has never seen during training,  
simply by assigning new parameter vectors to `vs.parameter_array`.

We estimate the transformer same way as we did before, but on disorder realizations is has not seen during training. For each of these new disorder realizations : we compute the exact ground state energy via lanczos, we calculate the ground energy and variance of the transformer for the given disorder with a FullSumState. This bloc returns the mean of the reel error and V-score on all disorder realizations

In [ ]:
N_test = 30
params_test = generate_disorder(N_test, hi.size, h0, rng=rng)  # new realizations

test_results = {"E_ed": [], "E_vmc": [], "rel_err": [], "v_score": []}

for r, pars in tqdm(enumerate(params_test), total=N_test, desc="Test"):
    _ha  = create_operator(pars)
    E_ed = nk.exact.lanczos_ed(_ha, k=1, compute_eigenvectors=False).item()

    _vs = vs.get_state(pars)   # ← same model weights, new disorder
    vs_fs = nk.vqs.FullSumState(
        hilbert    = hi,
        model      = _vs.model,
        variables  = _vs.variables,
        chunk_size = 64,
    )
    stats  = vs_fs.expect(_ha)
    E_vmc  = float(np.real(stats.Mean))
    Var    = float(np.real(stats.Variance))

    test_results["E_ed"].append(E_ed)
    test_results["E_vmc"].append(E_vmc)
    test_results["rel_err"].append(abs((E_vmc - E_ed) / E_ed))
    test_results["v_score"].append(Var / (E_vmc**2 + 1e-12))

for k_name in test_results:
    test_results[k_name] = np.array(test_results[k_name])

print(f"Test mean relative error : {test_results['rel_err'].mean()*100:.3f} %")
print(f"Test mean V-score        : {test_results['v_score'].mean():.2e}")

in this bloc we plot the exact energy against the ones expected by the transformer on new disorder realizations. We also plot the V-score of the transformer on each disorder realization 

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.scatter(test_results["E_ed"], test_results["E_vmc"], alpha=0.7, color="teal", s=40)
lims = [test_results["E_ed"].min() * 1.02, test_results["E_ed"].max() * 0.98]
ax.plot(lims, lims, "k--", linewidth=1)
ax.set_xlabel("Exact $E_0$")
ax.set_ylabel("VMC energy (test)")
ax.set_title("Generalization: VMC vs ED on unseen realizations")

ax = axes[1]
ax.bar(range(N_test), test_results["v_score"], color="teal", alpha=0.8)
ax.axhline(1e-3, color="red", linestyle="--")
ax.set_yscale("log")
ax.set_xlabel("Test replica")
ax.set_ylabel("V-score")
ax.set_title("V-score on test set")

fig.tight_layout()
plt.savefig("output/test_results.pdf")
plt.show()

---
## Part 9 — Summary and checklist

- Defined a transverse-field Ising Hamiltonian conditioned on a single random global field h0
- Generated N disorder realizations as a training set
- Built a ViTFNQS Foundational ansatz (disorder=True, transl_invariant=False)
- Ran joint VMC_NG optimization over N replicas simultaneously
- Monitored convergence via energy curves
- Checked MCMC mixing quality with R̂ (target: R̂ < 1.1)
- Measured variational quality with the V-score (target: V < 1e-3)
- Compared against exact Lanczos ED
- Evaluated generalization on unseen (test) disorder realizations


---

### Possible extensions

| Direction | How |
|-----------|-----|
| **Larger systems** | Increase `L`, use `nk.sampler.MetropolisExchange` or a `GlobalFlipRule` |
| **Multi-h₀ training** | Use `generate_multi_h0_disorder` to train across several disorder strengths simultaneously (cf. `1D_avec_desordre_pluri_h0.py`) |
| **2D disorder** | Set `two_dimensional=True`, `b` must satisfy `b² | L²` |
| **Phase transitions** | Sweep `J/h0` and plot disorder-averaged `⟨Mz²⟩` as an order parameter |
| **Better sampler** | Add a `GlobalFlipRule` with small probability to improve mixing in the ferromagnetic phase |

---

### Diagnostic quick-reference

| Metric | Formula | Good value | What to do if bad |
|--------|---------|------------|-------------------|
| **Relative energy error** | $|E_{VMC}-E_0|/|E_0|$ | $< 0.1\%$ | More layers, larger `d_model` |
| **V-score** | $\text{Var}[H]/E_0^2$ | $< 10^{-3}$ | More optimization steps, better LR |
| **R̂** | Gelman-Rubin | $< 1.05$ | More chains, longer burn-in |